In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from UCB_training.UCB_plotting import find_nan_gaps

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

PROJECT_ROOT = Path('../../')
RAW_DIR = PROJECT_ROOT / 'russian_river_data'

BASINS = ['calpella', 'hopland', 'guerneville', 'warm_springs']

# Column names in shared hourly.csv / daily.csv for observed flow
FLOW_COLS = {
    'calpella': 'NR CALPELLA FLOW COE CPL',
    'hopland': 'NR HOPLAND FLOW COE HOP',
    'guerneville': 'NR GUERNEVILLE FLOW COE GRN',
    'warm_springs': 'GEYSERVILLE CA FLOW USGS-MERGED',
}

SPLITS = {
    'Train': ('1994-10-01', '2002-09-30'),
    'Val': ('2002-10-01', '2005-09-30'),
    'Test': ('2005-10-01', '2009-09-29'),
}

SPLIT_COLORS = {'Train': '#2196F3', 'Val': '#FF9800', 'Test': '#4CAF50'}

### How the Raw Data is Parsed

The raw HMS export CSVs have 3 quirks that need handling:

1. **3 header rows**: row 0 = column numbers, row 1 = column names, row 2 = units
2. **`24:00:00` timestamps**: HMS convention for midnight — means `00:00:00` of the NEXT day
3. **Mixed date formats**: `1-Oct-94` style dates

Our `clean_df()` in `UCB_utils.py` handles all three. We replicate it here so the gap analysis
matches exactly what NeuralHydrology sees during training.

**Three types of gaps to check:**
- **Parse failures**: rows that can't be converted to datetime (after `24:00:00` fix)
- **Missing timestamps**: hours/days absent from the continuous date range
- **NaN values**: timestamp exists but sensor reading is missing

In [ ]:
def load_raw(path, freq='hourly'):
    """Replicate clean_df() parsing from UCB_utils.py"""
    raw = pd.read_csv(path, skiprows=[0, 2], header=0, low_memory=False)
    raw.columns = [c.strip() for c in raw.columns]
    date_col = [c for c in raw.columns if 'Date' in c][0]

    if freq == 'hourly':
        time_col = [c for c in raw.columns if 'Time' in c][0]
        # Handle 24:00:00 → next-day 00:00:00 (same as clean_df lines 63-68)
        mask_24 = raw[time_col].astype(str).str.strip() == '24:00:00'
        raw.loc[mask_24, date_col] = (
            pd.to_datetime(raw.loc[mask_24, date_col].str.strip(), format='%d-%b-%y')
            + pd.Timedelta(days=1)
        ).dt.strftime('%d-%b-%y')
        raw.loc[mask_24, time_col] = '00:00:00'
        n_24_fixed = mask_24.sum()

        raw['datetime'] = pd.to_datetime(
            raw[date_col].astype(str).str.strip() + ' ' + raw[time_col].astype(str).str.strip(),
            format='mixed', dayfirst=True, errors='coerce')
    else:
        n_24_fixed = 0
        raw['datetime'] = pd.to_datetime(raw[date_col].astype(str).str.strip(),
                                          format='mixed', dayfirst=True, errors='coerce')

    n_parse_fail = raw['datetime'].isna().sum()
    df = raw.dropna(subset=['datetime']).set_index('datetime').sort_index()

    # Numeric coercion (same as clean_df lines 76-77)
    for col in FLOW_COLS.values():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Check for duplicates
    n_dupes = df.index.duplicated().sum()

    print(f"File: {Path(path).name}")
    print(f"  Raw rows: {len(raw)} | 24:00→00:00 fixed: {n_24_fixed} | Parse failures: {n_parse_fail} | Dupes: {n_dupes}")
    print(f"  Final: {len(df)} rows | {df.index.min()} to {df.index.max()}")

    # Drop duplicates (keep first)
    df = df[~df.index.duplicated(keep='first')]
    return df

### Load Raw Data

In [ ]:
hourly = load_raw(RAW_DIR / 'hourly.csv', freq='hourly')

# Check missing timestamps
full_hourly = pd.date_range(hourly.index.min(), hourly.index.max(), freq='h')
missing_ts_hourly = full_hourly.difference(hourly.index)
print(f"\n  Expected timestamps: {len(full_hourly)}")
print(f"  Missing timestamps: {len(missing_ts_hourly)}")

In [ ]:
daily = load_raw(RAW_DIR / 'daily.csv', freq='daily')

full_daily = pd.date_range(daily.index.min(), daily.index.max(), freq='D')
missing_ts_daily = full_daily.difference(daily.index)
print(f"\n  Expected timestamps: {len(full_daily)}")
print(f"  Missing timestamps: {len(missing_ts_daily)}")

### Gap Detection

In [ ]:
def find_missing_ts_gaps(index, full_range):
    """Find contiguous blocks of missing timestamps."""
    missing = full_range.difference(index)
    if len(missing) == 0:
        return []
    gaps = []
    start = missing[0]
    prev = missing[0]
    for t in missing[1:]:
        if (t - prev) > (full_range.freq * 1.5):
            gaps.append((start, prev, len(pd.date_range(start, prev, freq=full_range.freq))))
            start = t
        prev = t
    gaps.append((start, prev, len(pd.date_range(start, prev, freq=full_range.freq))))
    return sorted(gaps, key=lambda x: -x[2])

### Complete Gap Report — Hourly

In [ ]:
for basin in BASINS:
    col = FLOW_COLS[basin]
    print(f"\n{'='*70}")
    print(f"  {basin.upper()} — HOURLY observed flow ({col})")
    print(f"{'='*70}")

    for split, (start, end) in SPLITS.items():
        sub = hourly.loc[start:end, col]
        full = pd.date_range(start, end, freq='h')
        present = sub.index
        missing_ts = full.difference(present)
        nan_count = sub.isna().sum()
        total_expected = len(full)
        total_missing = len(missing_ts) + nan_count
        pct = total_missing / total_expected * 100 if total_expected > 0 else 0

        print(f"\n  {split} ({start} to {end})")
        print(f"    Expected: {total_expected} hrs | Present: {len(sub)} | Missing timestamps: {len(missing_ts)} | NaN values: {nan_count}")
        print(f"    Total unusable: {total_missing} ({pct:.1f}%)")

        # NaN gaps > 1 day
        nan_gaps = find_nan_gaps(sub)
        big_nan = [(s, e, d) for s, e, d in nan_gaps if d > 24]
        if big_nan:
            print(f"    NaN gaps > 1 day ({len(big_nan)}):")
            for s, e, d in big_nan[:8]:
                print(f"      {s.strftime('%Y-%m-%d')} to {e.strftime('%Y-%m-%d')} ({d} hrs = {d//24} days)")

        # Missing timestamp gaps
        ts_gaps = find_missing_ts_gaps(present, full)
        big_ts = [(s, e, d) for s, e, d in ts_gaps if d > 24]
        if big_ts:
            print(f"    Missing timestamp gaps > 1 day ({len(big_ts)}):")
            for s, e, d in big_ts[:8]:
                print(f"      {s.strftime('%Y-%m-%d')} to {e.strftime('%Y-%m-%d')} ({d} hrs = {d//24} days)")

### Complete Gap Report — Daily

In [ ]:
for basin in BASINS:
    col = FLOW_COLS[basin]
    print(f"\n{'='*70}")
    print(f"  {basin.upper()} — DAILY observed flow")
    print(f"{'='*70}")

    for split, (start, end) in SPLITS.items():
        sub = daily.loc[start:end, col]
        full = pd.date_range(start, end, freq='D')
        missing_ts = full.difference(sub.index)
        nan_count = sub.isna().sum()
        total_expected = len(full)
        total_missing = len(missing_ts) + nan_count
        pct = total_missing / total_expected * 100 if total_expected > 0 else 0

        print(f"\n  {split} ({start} to {end})")
        print(f"    Expected: {total_expected} days | Present: {len(sub)} | Missing timestamps: {len(missing_ts)} | NaN values: {nan_count}")
        print(f"    Total unusable: {total_missing} ({pct:.1f}%)")

        nan_gaps = find_nan_gaps(sub)
        big_nan = [(s, e, d) for s, e, d in nan_gaps if d > 1]
        if big_nan:
            print(f"    NaN gaps > 1 day ({len(big_nan)}):")
            for s, e, d in big_nan[:8]:
                print(f"      {s.strftime('%Y-%m-%d')} to {e.strftime('%Y-%m-%d')} ({d} days)")

### Data Availability Timeline

In [ ]:
def plot_availability(df, freq_label='hourly', figsize=(16, 5)):
    basins = list(FLOW_COLS.keys())
    fig, axes = plt.subplots(len(basins), 1, figsize=figsize, sharex=True)
    t0, t1 = df.index.min(), df.index.max()

    for ax, basin in zip(axes, basins):
        col = FLOW_COLS[basin]
        # Green bar = full date range (available)
        ax.broken_barh([(t0, t1 - t0)], (0.2, 0.6), facecolor='#4CAF50', edgecolor='none')

        # Red bars = NaN gaps
        gaps = find_nan_gaps(df[col])
        for gs, ge, dur in gaps:
            ax.broken_barh([(gs, ge - gs + pd.Timedelta(hours=1))], (0.2, 0.6),
                           facecolor='#E53935', edgecolor='none')

        # Split dividers
        for split, (start, end) in SPLITS.items():
            s = pd.Timestamp(start)
            ax.axvline(s, color='white', lw=1.2, zorder=3)
            ax.text(s + pd.Timedelta(days=90), 0.85, split, fontsize=9,
                    color=SPLIT_COLORS[split], fontweight='bold', ha='left',
                    bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.7))

        ax.set_yticks([])
        ax.set_ylabel(basin.replace('_', ' ').title(), fontsize=10, rotation=0,
                      ha='right', va='center')
        ax.set_ylim(0, 1)
        ax.set_xlim(t0, t1)

    axes[-1].xaxis.set_major_locator(mdates.YearLocator())
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    fig.suptitle(f'Observed Flow Data Availability ({freq_label})', fontsize=13, y=1.01)

    # Legend
    from matplotlib.patches import Patch
    axes[0].legend(handles=[Patch(fc='#4CAF50', label='Available'),
                            Patch(fc='#E53935', label='Missing / NaN')],
                   loc='upper right', fontsize=8, framealpha=0.9)
    plt.tight_layout()
    plt.show()

plot_availability(hourly, 'hourly')

In [ ]:
plot_availability(daily, 'daily', figsize=(16, 6))

### Unusable Data Percentage (NaN + Missing Timestamps)

In [ ]:
rows = []
for basin in BASINS:
    col = FLOW_COLS[basin]
    for freq_label, df in [('hourly', hourly), ('daily', daily)]:
        freq_code = 'h' if freq_label == 'hourly' else 'D'
        for split, (start, end) in SPLITS.items():
            sub = df.loc[start:end, col]
            full = pd.date_range(start, end, freq=freq_code)
            missing_ts = len(full.difference(sub.index))
            nan_vals = sub.isna().sum()
            total = len(full)
            rows.append({
                'basin': basin.replace('_', ' ').title(), 'freq': freq_label, 'split': split,
                'missing_ts': missing_ts, 'nan_vals': nan_vals,
                'total_unusable_pct': round((missing_ts + nan_vals) / total * 100, 1) if total > 0 else 0
            })

gap_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, freq in zip(axes, ['hourly', 'daily']):
    sub = gap_df[gap_df['freq'] == freq].pivot(index='basin', columns='split', values='total_unusable_pct')
    sub = sub[['Train', 'Val', 'Test']]
    sns.heatmap(sub, annot=True, fmt='.1f', cmap='YlOrRd', vmin=0, vmax=35, ax=ax,
                cbar_kws={'label': 'Unusable %'})
    ax.set_title(f'{freq.title()} — % Unusable (NaN + Missing)')
    ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
print("DETAILED BREAKDOWN: Missing Timestamps vs NaN Values\n")
detail = gap_df.pivot_table(index=['basin', 'freq'], columns='split',
                            values=['missing_ts', 'nan_vals', 'total_unusable_pct'])
print(detail[['missing_ts', 'nan_vals', 'total_unusable_pct']].to_string())

### Shared vs Basin-Specific Gaps

In [ ]:
print("Which gaps are shared across basins?\n")
for split, (start, end) in SPLITS.items():
    print(f"--- {split} ({start} to {end}) ---")
    nan_masks = {}
    for basin in BASINS:
        col = FLOW_COLS[basin]
        sub = hourly.loc[start:end, col]
        nan_masks[basin] = sub.isna()

    shared_3 = nan_masks['calpella'] & nan_masks['hopland'] & nan_masks['guerneville']
    shared_all = shared_3 & nan_masks['warm_springs']
    only_3 = shared_3 & ~nan_masks['warm_springs']

    print(f"  All 4 basins NaN: {shared_all.sum()} hrs ({shared_all.sum()//24} days)")
    print(f"  3 basins (excl. warm_springs): {only_3.sum()} hrs ({only_3.sum()//24} days)")

    for basin in BASINS:
        unique = nan_masks[basin] & ~shared_3
        if unique.sum() > 0:
            print(f"  {basin}-only NaN: {unique.sum()} hrs ({unique.sum()//24} days)")
    print()

### Implications for Model Evaluation

Key takeaways from the gap analysis:
- Metrics are computed ONLY on non-NaN observed timesteps
- Basins with more gaps have fewer evaluation points, potentially biasing metric comparisons
- Seasonal bias: if gaps concentrate in dry season (summer), metrics overweight wet season performance

In [ ]:
print("EFFECTIVE EVALUATION POINTS (hourly test period)\n")
for basin in BASINS:
    col = FLOW_COLS[basin]
    sub = hourly.loc['2005-10-01':'2009-09-29', col]
    full = pd.date_range('2005-10-01', '2009-09-29', freq='h')
    valid = sub.dropna()
    print(f"  {basin:15s}: {len(valid):6d} / {len(full)} usable ({len(valid)/len(full)*100:.1f}%)")

    # Check seasonal distribution of valid data
    valid_months = valid.index.month
    wet = valid_months.isin([10, 11, 12, 1, 2, 3]).sum()
    dry = valid_months.isin([4, 5, 6, 7, 8, 9]).sum()
    print(f"                   Wet season (Oct-Mar): {wet} ({wet/len(valid)*100:.0f}%)  |  Dry season (Apr-Sep): {dry} ({dry/len(valid)*100:.0f}%)")